# Step 10a — MVT Grid: CIFAR-FS, 5-shot (RUN FIRST)

**Settings:** GPU T4, Internet ON, attach the Kaggle dataset `beft-thesis-data`
(owner `notavailable73` — note the slug is missing the "p" in "bpeft", a
pre-existing typo in how the dataset was created; the display title still
reads "bpeft-thesis-data"). See `step_writeups/step10.txt` for the full
reasoning and `plan.md` for the grid design. This notebook only drives
`scripts/run_mvt_grid.py --only "dataset=cifar_fs,shots=5"`; every other Step 10 script (config
generation, aggregation, tables, plots) is shared across all three notebooks
and documented there — nothing new lives in this notebook itself.


## 1. GPU check + clone repo + install deps

In [ ]:
import torch, sys, os, subprocess
print('python:', sys.version.split()[0], '| torch:', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU T4'

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/thesis'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repo ready at', os.getcwd())


## 2. Stage data — symlink the attached dataset into `data/`

Nothing here is a hard requirement: every dataset falls back to a runtime download if its symlink source is missing (Internet must be ON).

In [ ]:
# Attach the Kaggle dataset `beft-thesis-data` (owner notavailable73) before
# running this cell: https://www.kaggle.com/datasets/notavailable73/beft-thesis-data
# Its real upload structure (verified via the Kaggle API, 2026-08-03) is:
#   bpeft-data/cifar-100-python/{meta,train,test}
#   bpeft-data/svhn/test_32x32.mat
#   bpeft-data/tinyimagenet/tiny-imagenet-200/tiny-imagenet-200/{train,val,test,wnids.txt,...}
#   bpeft-data/miniimagenet/mini-imagenet-cache-{train,validation,test}.pkl
# (Zenodo pkl caches, not the .npy/.json format plan.md Section 4.2 originally
# assumed -- src/datasets/mini_imagenet.py already supports this layout
# natively, so nothing needed regenerating.) Rather than hardcode that exact
# nesting, this reuses the SAME staged-path finder functions
# scripts/train.py / evaluate.py call at runtime, so it is guaranteed to
# symlink to whatever those modules would discover themselves -- correct
# regardless of whether Kaggle mounts this dataset one level deeper
# (`/kaggle/input/datasets/<owner>/<slug>/...`) or double-wraps the
# tiny-imagenet-200 folder on auto-unzip (both observed live in earlier
# steps; see each finder's docstring).
import os, shutil

from src.datasets.cifar_fs import _find_staged_cifar100_root
from src.datasets.svhn_ood import _find_staged_svhn_root
from src.datasets.tinyimagenet_ood import _find_extracted_tin_root
from src.datasets.mini_imagenet import _find_zenodo_pkls

LINKS = {}

cifar100_root = _find_staged_cifar100_root('data')
if cifar100_root:
    LINKS['data/cifar-100-python'] = os.path.join(cifar100_root, 'cifar-100-python')

svhn_root = _find_staged_svhn_root('data')
if svhn_root:
    LINKS['data/svhn/test_32x32.mat'] = os.path.join(svhn_root, 'test_32x32.mat')

tin_root = _find_extracted_tin_root('data')
if tin_root:
    LINKS['data/tiny-imagenet-200'] = tin_root

for split, pkl_path in (_find_zenodo_pkls('data') or {}).items():
    LINKS[f'data/{pkl_path.name}'] = str(pkl_path)

os.makedirs('data/svhn', exist_ok=True)
if not LINKS:
    print('No staged files found under /kaggle/input -- did you attach '
          '`beft-thesis-data`? Falling back to runtime downloads for everything.')
for link, target in LINKS.items():
    if os.path.exists(link) or os.path.islink(link):
        print(f'OK   (already present): {link}')
        continue
    if not os.path.exists(target):
        print(f'MISSING source -- {os.path.basename(link)} will fall back to '
             f'runtime download (target not found: {target})')
        continue
    try:
        os.symlink(target, link)
        print(f'OK   (symlinked): {link} -> {target}')
    except OSError as e:
        print(f'symlink failed ({e}); copying instead (slower): {link}')
        (shutil.copytree if os.path.isdir(target) else shutil.copy2)(target, link)
        print(f'OK   (copied): {link}')


## 3. Build the frozen splits (CIFAR-FS + MiniImageNet)

In [ ]:
!python scripts/build_cifar_fs_split.py
!python scripts/build_mini_imagenet_split.py


## 4. Generate the 120 grid configs + run the offline config tests

In [ ]:
!python scripts/build_grid_configs.py
!python -m pytest -q tests/test_grid_configs.py


## 5. Pre-flight smoke test (run once; the 3 genuinely-new risks)

In [ ]:
# Pre-flight smoke test (plan.md Section 2) -- the three genuinely-new risks:
# 1-shot (never run), LoRA x MobileNetV3-Small (never run), LoRA x MiniImageNet
# (never run). ~10 min total. Run once; safe to skip on later sessions since
# --resume below will re-skip these same cells once the real grid overwrites
# their results (set RUN_SMOKE_TEST = False after the first successful pass).
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    SMOKE_ONLY = [
        'dataset=cifar_fs,shots=1,backbone=r18,adapter=bottleneck_parallel,head=evidential,seed=42',
        'dataset=cifar_fs,shots=5,backbone=mbnet,adapter=lora,head=evidential,seed=42',
        'dataset=mini_imagenet,shots=5,backbone=r18,adapter=lora,head=evidential,seed=42',
    ]
    for only in SMOKE_ONLY:
        !python scripts/run_mvt_grid.py --only "{only}" --num-episodes 20 --wandb-mode disabled
    print('smoke test done -- inspect the [grid] lines above for status=ok on all three.')
else:
    print('RUN_SMOKE_TEST is False -- skipping (already run in a prior session).')


## 6. Log in to W&B + run the grid

Add a Kaggle Secret named `WANDB_API_KEY` (notebook editor → Add-ons →
Secrets; get the key from <https://wandb.ai/authorize>) before running this
cell so the grid's runs upload online and group by (dataset, shots) per
`progress.txt`'s Step 10 exit criteria. Falls back to **offline** mode
(writes to `./wandb/`, sync later with `wandb sync wandb/`) if the secret is
missing or login fails — this cell never calls interactive `wandb.login()`,
so the unattended `--max-minutes 660` run below can't stall waiting on
stdin. Login and the grid launch are ONE cell on purpose (a prior version
split them across two cells and passed `WANDB_MODE` to a separate `!`
shell cell via `--wandb-mode {WANDB_MODE}`; that silently broke if the two
cells were ever run out of order or after a kernel restart, since IPython
leaves an unresolved `{name}` in a `!` command as literal text instead of
erroring). Resumable: re-running this cell after a session timeout picks up
where it left off (`--resume` skips any cell whose results JSON already
exists).

In [ ]:
import os
import subprocess
import sys

import wandb

api_key = os.environ.get("WANDB_API_KEY")
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if api_key:
            os.environ["WANDB_API_KEY"] = api_key
            print("Loaded WANDB_API_KEY from Kaggle Secrets.")
    except Exception as e:
        print(f"Kaggle Secrets lookup skipped: {e!r}")

if not api_key:
    print("No WANDB_API_KEY found (env or Kaggle Secrets) -- using offline "
          "mode so this run never blocks on an interactive login prompt.")
    WANDB_MODE = "offline"
else:
    WANDB_MODE = "online"
    try:
        if not wandb.login(key=api_key):
            print("wandb.login() returned False -- falling back to offline mode.")
            WANDB_MODE = "offline"
    except Exception as e:
        print(f"wandb.login() failed: {e!r} -- falling back to offline mode.")
        WANDB_MODE = "offline"

print("WANDB_MODE for this session:", WANDB_MODE)

cmd = [sys.executable, "scripts/run_mvt_grid.py",
       "--resume", "--only", "dataset=cifar_fs,shots=5",
       "--max-minutes", "660", "--use-tinyimagenet", "--use-gaussian",
       "--wandb-mode", WANDB_MODE]
print(">>>", " ".join(cmd))
subprocess.run(cmd)


## 7. Pack + push this notebook's slice of results

In [ ]:
# Pack + push this notebook's slice only (mirrors Step 9 notebook Section 9/9b).
import glob, hashlib, json as _json, os, subprocess, zipfile

ARTIFACT_STEM = 'step10a_cifar_5shot'
RUN_TAG_GLOB = 'grid_cifar_5shot'

ZIP = f'/kaggle/working/{ARTIFACT_STEM}_artifacts.zip'
RESULTS = sorted(glob.glob(f'results/grid/*{RUN_TAG_GLOB}*'))
LOGS = ['results/grid/_run_log.jsonl'] if os.path.exists('results/grid/_run_log.jsonl') else []
CHECKPOINTS = sorted(glob.glob(f'checkpoints/model_phase2_{RUN_TAG_GLOB}*.pt'))
ALL_FILES = RESULTS + LOGS + CHECKPOINTS

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in ALL_FILES:
        zf.write(p)
    manifest_lines = [
        f'{p}  {os.path.getsize(p)}B  sha256={hashlib.sha256(open(p, "rb").read()).hexdigest()}'
        for p in ALL_FILES
    ]
    zf.writestr('MANIFEST.txt', '\n'.join(manifest_lines))
print(f'wrote {ZIP} ({len(ALL_FILES)} files)')

# Channel 1: push to a Kaggle dataset (survives the browser tab closing) if
# KAGGLE_USERNAME / KAGGLE_KEY secrets are configured; channel 2: browser
# download otherwise. See notebooks/step9-mini(1).ipynb Section 9b for the
# fuller version of this pattern this is condensed from.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')
    HAVE_SECRETS = True
except Exception:
    HAVE_SECRETS = False

if HAVE_SECRETS:
    ds_dir = f'/kaggle/working/{ARTIFACT_STEM}_dataset'
    os.makedirs(ds_dir, exist_ok=True)
    subprocess.run(['cp', ZIP, ds_dir], check=True)
    meta = {
        "title": f"{ARTIFACT_STEM}-artifacts",
        "id": f"{os.environ['KAGGLE_USERNAME']}/{ARTIFACT_STEM}-artifacts",
        "licenses": [{"name": "CC0-1.0"}],
    }
    _json.dump(meta, open(f'{ds_dir}/dataset-metadata.json', 'w'))
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', ds_dir, '-q'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run(['kaggle', 'datasets', 'version', '-p', ds_dir, '-m', 'update', '-q'])
    print(f'pushed to Kaggle dataset {ARTIFACT_STEM}-artifacts')
else:
    from IPython.display import FileLink, display
    display(FileLink(ZIP))
    print('no Kaggle Secrets found -- use the download link above instead.')


## After this session

1. Do **not** run `scripts/aggregate_grid.py` / `make_master_tables.py` /
   `grid_plots.py` here — those run ONCE, locally or on plain CPU, after
   10a + 10b + 10c have ALL finished and their artifact zips are merged into
   one local `results/grid/` (plan.md Section 10.4).
2. Note any `collapsed` or `error` status lines from Section 6's output, and
   `results/grid/_run_log.jsonl`'s `best_val_epoch` values, for
   `step_writeups/step10.txt`.
